## Testing Embeddings

In [1]:
from FlagEmbedding import FlagModel 
import os

os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = 'true'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

d:\University\FCSE\Courses\26S\TUG-26S-BT-Content_Optimization\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = FlagModel('BAAI/bge-base-en-v1.5')

sentences = [
    "That is a happy dog",
    "That is a very happy person",
    "Today is a sunny day",
]

embeddings = model.encode(sentences)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 410.13it/s]


In [3]:
print(f"Embeddings:\n{embeddings.shape}")

Embeddings:
(3, 768)


In [4]:
scores = embeddings @ embeddings.T
print(f"Similiarty scores:\n{scores}")

Similiarty scores:
[[1.0000001 0.7900388 0.5752543]
 [0.7900388 0.9999999 0.5919022]
 [0.5752543 0.5919022 0.9999999]]


## Testing File Things

In [4]:
import json

In [ ]:
from pathlib import Path


def read_file_contents(file_path):
    file_contents = ""
    with open(file_path, 'r', encoding='utf-8') as f:
        file_contents += f.read()
    return file_contents


with open('../output/wikipedia-pets/wikipedia-pets/page_index.json', mode='r', encoding='utf-8-sig') as f:
    for entry in json.load(f):
        try: 
            curr_file = entry["text_file"].replace('texts/', '')
            curr_page_id = entry["page_id"]
            open("../data/corpus_cleaned/" + curr_file, 'r', encoding='utf-8')

        except AttributeError as e:
            fail_counter += 1
            print(f"Skipping 404 - {curr_file}")
        

IndentationError: expected an indented block after 'except' statement on line 22 (3052978576.py, line 22)

## Actual Script

In [ ]:
import csv
import json

In [6]:
from FlagEmbedding import FlagModel 
import os

os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = 'true'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

model = FlagModel('BAAI/bge-base-en-v1.5')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1797.23it/s]


In [7]:
def read_file_contents(file_path):
    file_contents = ""
    with open(file_path, 'r', encoding='utf-8') as f:
        file_contents += f.read()
    return file_contents

In [23]:
import spacy

nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")

In [25]:
def smart_text_chunking(text, chunk_size=512):
    doc = nlp(text)
    sentences = [sent.text for sent in doc.sents]

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        sentence_words = len(sentence.split())
        current_words = len(current_chunk.split())

        if sentence_words > chunk_size:
            raise ValueError(f"Sentence is too long to fit in a chunk: {sentence[:200]}...")

        if sentence_words + current_words <= chunk_size:
            current_chunk += " " + sentence
        else:
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
            current_chunk = sentence

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path


REL_CHUNK_METADATA_FILE = Path("../data/embeddings/chunk_metadata.parquet")
REL_CHUNK_EMBEDDINGS_FILE = Path("../data/embeddings/chunk_embeddings.npy")
REL_PAGE_INDEX_FILE = Path("../output/wikipedia-pets/page_index.json")


def build_chunk_metadata(corpus_dir, page_index_file=REL_PAGE_INDEX_FILE, chunk_size=512):
    corpus_dir = Path(corpus_dir)
    rows = []
    skipped_pages = []
    global_chunk_id = 0
    page_counter = 0

    with open(page_index_file, mode="r", encoding="utf-8-sig") as f:
        for entry in json.load(f):
            if page_counter % 100 == 0:
                print(f"Progress Update: [{page_counter}/21077]")
            text_file = entry.get("text_file")
            if not text_file:
                skipped_pages.append({
                    "page_id": entry.get("page_id"),
                    "title": entry.get("title"),
                    "reason": "missing text_file",
                })
                continue

            curr_file = text_file.replace("texts/", "")
            file_path = corpus_dir / curr_file

            if not file_path.exists():
                skipped_pages.append({
                    "page_id": entry.get("page_id"),
                    "title": entry.get("title"),
                    "file_name": curr_file,
                    "reason": "missing corpus file",
                })
                continue

            try:
                text = read_file_contents(file_path)
                curr_chunks = smart_text_chunking(text, chunk_size=chunk_size)
            except ValueError as e:
                skipped_pages.append({
                    "page_id": entry.get("page_id"),
                    "title": entry.get("title"),
                    "file_name": curr_file,
                    "reason": str(e),
                })
                continue

            for chunk_index, chunk_text in enumerate(curr_chunks):
                chunk_text = chunk_text.strip()
                if not chunk_text:
                    continue

                rows.append({
                    "global_chunk_id": global_chunk_id,
                    "page_id": entry["page_id"],
                    "title": entry.get("title", ""),
                    "file_name": curr_file,
                    "chunk_index": chunk_index,
                    "word_count": len(chunk_text.split()),
                    "char_count": len(chunk_text),
                    "text": chunk_text,
                })
                global_chunk_id += 1
            
            page_counter += 1

    chunk_metadata = pd.DataFrame(rows)
    skipped_pages = pd.DataFrame(skipped_pages)
    return chunk_metadata, skipped_pages


def save_chunk_metadata_parquet(corpus_dir="../data/corpus_cleaned/", output_file=REL_CHUNK_METADATA_FILE):
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    chunk_metadata, skipped_pages = build_chunk_metadata(corpus_dir)
    chunk_metadata.to_parquet(output_file, index=False)

    print(f"Saved {len(chunk_metadata)} chunks to {output_file}")
    print(f"Skipped {len(skipped_pages)} pages")
    return chunk_metadata, skipped_pages


In [ ]:
def encode_chunks_from_metadata(chunk_metadata, output_file=REL_CHUNK_EMBEDDINGS_FILE):
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    texts = chunk_metadata.sort_values("global_chunk_id")["text"].tolist()
    embeddings = model.encode(texts)
    np.save(output_file, embeddings)

    print(f"Saved embeddings with shape {embeddings.shape} to {output_file}")
    return embeddings

In [19]:
chunk_metadata, skipped_pages = save_chunk_metadata_parquet("../data/corpus_cleaned/")
# embeddings = encode_chunks_from_metadata(chunk_metadata)# Run manually when ready:
# chunk_metadata, skipped_pages = save_chunk_metadata_parquet("../data/corpus_cleaned/")
# embeddings = encode_chunks_from_metadata(chunk_metadata)

Progress Update: [0/21077]
Progress Update: [100/21077]
Progress Update: [200/21077]
Progress Update: [300/21077]
Progress Update: [400/21077]
Progress Update: [500/21077]
Progress Update: [600/21077]


KeyboardInterrupt: 

## Manual Extra

In [ ]:
REL_CHUNK_INFORMATION_FILE = "../data/embeddings/chunk_information.csv"
REL_PAGE_INDEX_FILE = "../output/wikipedia-pets/wikipedia-pets/page_index.json"

def get_file_chunks(corpus_dir):
    # Matching chunking to embeddings.  
    # - Read the JSON from output/wikipedia_pets/page_index.json. 
    # - Read article file_name, and match it to page_index.json's text_file field without the texts/ prefix. 
    # - Match chunks with files via page_id, so we can track which embeddings belong to which files. 

    #all_article_chunks = []

    # Because script is very slow, we have to track how far we reached in the previous write session before we continue with this one. 
    # - This means first opening the file in read mode and reading the last line to see the page id of the last written file. 
    # - Then saving the last page id and chunk_id, and proceeding from there, rather than writing evrything again. 
    last_done_id = ""
    try: 
        with open(REL_CHUNK_INFORMATION_FILE, mode='r', encoding='utf-8') as f: 
            for line in csv.reader(f): 
                
    except FileNotFoundError as e: 
        print("No chunk data, starting from the beggining.")
        

    # page_id, chunk_id, chunk_text
    chunk_info_file = open(REL_CHUNK_INFORMATION_FILE, mode='w', encoding='utf-8')
    chunk_info_writer = csv.writer(chunk_info_file)

    with open(REL_PAGE_INDEX_FILE, mode='r', encoding='utf-8-sig') as f:
        for entry in json.load(f):
            try: 
                curr_file = entry["text_file"].replace('texts/', '')
                curr_page_id = entry["page_id"]
                open(f"{corpus_dir}{curr_file}" , 'r', encoding='utf-8')
                curr_chunks = smart_text_chunking(read_file_contents(f"{corpus_dir}{curr_file}"))
                if len(curr_chunks) == 1:
                    # page_id, chunk_id, chunk_text
                    chunk_info_writer.writerow([curr_page_id, 0, curr_chunks[0]])
                else:
                    for i, chunk in enumerate(curr_chunks):
                        chunk_info_writer.writerow([curr_page_id, i, chunk])
                # all_article_chunks.extend(curr_chunks)
                
            except AttributeError as e:
                print(f"Skipping 404 - {curr_file}")

            except ValueError as e:
                print (f"Skipping sentence that is too long in {curr_file}")
            
    return all_article_chunks

In [ ]:
with open("..\output\wikipedia-pets\page_index.json", mode="r", encoding="utf-8-sig") as f:
    counter = 0
    for json_entry in json.load(f):
        counter += 1
    
    print(counter)    

21077
